<a href="https://colab.research.google.com/github/istiaquehussain/ML-Staging/blob/main/Multimodal_Rag_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf pandas PyPDF2 langchain langchain_community langchain_core google.generativeai langchain-google-genai langchain-chroma>=0.1.2

In [2]:
import fitz  # PyMuPDF
import pandas as pd
import PyPDF2
import os
import time
from io import StringIO
from collections import defaultdict
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain.schema import Document
from typing import List
from langchain.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import LLMChain
from langchain_core.runnables import RunnableLambda
from langchain.schema import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from uuid import uuid4
from IPython.display import display, HTML

**FileUtil** class used for various file operations, including converting file paths to folder paths,
    creating directories, finding files with specific extensions, and creating text and image files.
    

In [3]:
class FileUtil:
  def __init__(self):
        pass

  def get_folder_equivalent_of_file_path(self,file_path,verbose=False):
      file_path.replace(" ","_")
      base_name = os.path.basename(file_path)
      file_name = os.path.splitext(base_name)[0]
      file_path = os.path.dirname(file_path)
      equivalent_file_path = file_path+'/'+file_name+"/"
      if verbose:
        print(f"get_folder_equivalent_of_file_path equivalent file path: {equivalent_file_path}")
      return equivalent_file_path

  def create_folder(self,file_path,verbose=False):
      directory = os.path.dirname(file_path)
      if verbose:
        print(f"Creating directory: {directory}")


      # Create the directory if it doesn't exist
      if not os.path.exists(directory):
          os.makedirs(directory)
          if verbose:
            print(f"Directory '{directory}' created.")
      else:
          if verbose:
            print(f"Directory '{directory}' already exists.")

  def create_folder_equivalent_of_file_path(self,file_path,verbose=False)->str:
    folder_path=self.get_folder_equivalent_of_file_path(file_path,verbose)
    self.create_folder(folder_path,verbose)
    return folder_path

  def find_files_with_extensions(self,extensions, folder_path,verbose=False):
      # Initialize a defaultdict to store lists of file paths by extension
      files_by_extension = defaultdict(list)

      # Walk through the directory
      for dirpath, _, filenames in os.walk(folder_path):
          for file_name in filenames:
              # Get the file extension
              file_ext = os.path.splitext(file_name)[1].lower()
              # Check if the file has one of the specified extensions
              if file_ext in [ext.lower() for ext in extensions]:
                  # Get the absolute path of the file
                  absolute_path = os.path.join(dirpath, file_name)
                  # Add the file path to the corresponding extension's list
                  files_by_extension[file_ext].append(absolute_path)

      file_extention_with_paths:dict=dict(files_by_extension)
      if verbose:
        print(f"find_files_with_extensions generated dict: {file_extention_with_paths}")
      return file_extention_with_paths

  def create_text_file(self,text, output_folder,verbose=False):
      output_file = output_folder + '/text.txt'
      with open(output_file, 'w', encoding='utf-8') as file:
          file.write(text)
      if verbose:
        print(f"create_text_file: text file created as {output_file}")


  def create_image_files(self,images, output_folder,verbose=False):
      for i, image in enumerate(images):
          output_file_temp = output_folder + f'/image_{i}.png'
          with open(output_file_temp, 'wb') as img_file:
            img_file.write(image)
      if verbose:
        print(f"create_image_files: image files created at {output_folder}")



**Extractor** is a utility class for extracting content from PDF files, including text and images.
    It also provides methods to save extracted content and list files by extension.
  

In [4]:
class Extractor:

  def __init__(self,file_util):
    self.file_util=file_util

  # Function to extract text
  def extract_text(self,pdf_file,verbose=False):
      with open(pdf_file, 'rb') as file:
          pdf_reader = PyPDF2.PdfReader(file)
          total_pages=len(pdf_reader.pages)
          text = ''
          #for page_num in range(pdf_reader.numPages):
          for page_num in range(total_pages):
              #page = pdf_reader.getPage(page_num)
              page = pdf_reader.pages[page_num]
              text += page.extract_text()
      if verbose:
        print(f"extract_text: text extaction completed for {pdf_file}")
      return text

  # Function to extract images
  def extract_images(self,pdf_file,verbose=False):
      pdf_document = fitz.open(pdf_file)
      images = []
      for page_num in range(len(pdf_document)):
          page = pdf_document.load_page(page_num)
          image_list = page.get_images(full=True)
          for image_index, img in enumerate(page.get_images(full=True)):
              xref = img[0]
              base_image = pdf_document.extract_image(xref)
              image_bytes = base_image["image"]
              images.append(image_bytes)
      if verbose:
        print(f"extract_images: image extaction completed for {pdf_file}")
      return images

  def extract_pdf_content(self,pdf_file_absoute_path,verbose=False):
      #extract text content
      text_content = self.extract_text(pdf_file_absoute_path,verbose)
      #extract images
      images = self.extract_images(pdf_file_absoute_path,verbose)
      #create folder to store extracted content and images
      temp_extract_folder=self.file_util.create_folder_equivalent_of_file_path(pdf_file_absoute_path,verbose)
      #store text content as text.txt files
      self.file_util.create_text_file(text_content, temp_extract_folder,verbose)
      #store all images as files
      self.file_util.create_image_files(images, temp_extract_folder,verbose)
      return temp_extract_folder

  def find_files_with_extensions(self,extensions,folder_path,verbose=False):
      temp_extract_folder=self.file_util.find_files_with_extensions(extensions,folder_path,verbose)
      return temp_extract_folder

  def extract_pdf_content_and_list_files(self,pdf_file_absoute_path,list_of_extensions,verbose=False):
      temp_extract_folder=self.extract_pdf_content(pdf_file_absoute_path,verbose)
      files_by_extension=self.find_files_with_extensions(list_of_extensions,temp_extract_folder,verbose)
      return files_by_extension

  def extract_text_file_into_chuncks(self,text_file_path_with_name:str)->List[Document]:
      loader = TextLoader(text_file_path_with_name)
      text_splitter = CharacterTextSplitter(
          separator=".",
          chunk_size=500,
          chunk_overlap=50,
          length_function=len,
          is_separator_regex=False,
      )
      documents = loader.load()
      documents = text_splitter.split_documents(documents)
      #text_chunks = [document.page_content for document in documents]
      return documents


**Summarizer** is a utility class for summarizing text and images using language models (LLMs).
    Provides methods to generate text summaries, create image descriptions, and combine
    text and image data for query-based summarization.
   

In [5]:
class Summarizer:

  def __init__(self,chat_llm,image_llm):
    self.chat_llm=chat_llm
    self.image_llm=image_llm

  def generate_text_summaries(self,documents:List[Document],hasLimittedLLMAccess=True,verbose=False):
    text_summarize_prompt="""You are an assistant tasked with summarizing text and table for retrival. \
    These summraries will be embedded and used to rertieve the raw text or table elements. \
    Give a consise summary of the text or the table that is well optimised for retrival. Text or table: {element}"""
    prompt_template = PromptTemplate(
        input_variables=["element"],
        template=text_summarize_prompt
    )
    llm_chain = prompt_template | self.chat_llm | StrOutputParser()
    shoudDelay=False
    if hasLimittedLLMAccess and len(documents)>14:
      shoudDelay=True
    counter=0
    for document in documents:
      counter = counter+1
      try:
        if shoudDelay and counter>13:
          print("delaying 70 sec due to LLM resource limitation")
          counter=0
          time.sleep(70)
        reponse = llm_chain.invoke({"element": document.page_content})
      except Exception as e:
        print(f"delaying 70 sec due to exception {e}")
        time.sleep(70)
        reponse = llm_chain.invoke({"element": document.page_content})
      document.page_content=reponse
    if verbose:
      print(documents)
    return documents

  def get_image_description(self,absolute_image_path:str,image_summary_promt)->dict:
      message_dict=self.create_message_dict(absolute_image_path,image_summary_promt)
      image_description=self.image_llm.invoke(message_dict).content
      return {"uri":absolute_image_path,"image_decription":image_description}

  def create_message_dict(self,absolute_image_path,image_summary_promt):
      message_dict_prompt={"type":"text","text":image_summary_promt}
      message_dict_image={"type":"image_url","image_url":absolute_image_path}
      messages=HumanMessage(content=[message_dict_prompt,message_dict_image])
      return [messages]

  def generate_image_summaries(self,image_paths:List[str],hasLimittedLLMAccess=True,verbose=False):
    image_summary_promt="""You are an assistant tasked with describing images for retrieval. \
  These detailed description will be embedded and used to retrieve the raw image . \
  Give a detailed description of the image that is well optimised for retrieval.\
  if the image looks like table give datail of each rows and cell \
  also inlude  Table title and Column headers"""
    image_summaries=[]
    shoudDelay=False
    if hasLimittedLLMAccess and len(image_paths)>14:
      shoudDelay=True
    counter=0
    for image_path in image_paths:
      if verbose:
        print(f"processing image {image_path}")
      counter = counter+1
      try:
        image_decription = self.get_image_description(image_path,image_summary_promt)
        shoudDelay = shoudDelay and counter>13
        if shoudDelay or image_decription.get('image_decription') == 'Error Processing document':
          print("delaying 70 sec due to LLM resource limitations")
          counter=0
          time.sleep(80)
        image_decription = self.get_image_description(image_path,image_summary_promt)
        image_summaries.append(image_decription)
      except Exception as e:
        print(f"delaying 70 sec due to excetion {e}")
        time.sleep(70)
        image_decription = self.get_image_description(image_path,image_summary_promt)
        image_summaries.append(image_decription)

    douments=[]
    for image_summary in image_summaries:
      document=Document(page_content=image_summary.get('image_decription'),metadata={"image_url": image_summary.get('uri')})
      douments.append(document)
    if verbose:
      print(douments)
    return douments

  def summarize(self,query,documents,verbose=False):
    douments_str= "\n".join(document.page_content for document in documents)
    llm_summary_promt=f"""You are an assistant tasked with chat agent.\nYou will be given text , tables and image(s) data which usually chart or graph.\nUse this information to answer the user provided question.\nUser provided question is: \n {query}\nText and /or tables and /or images  are: \n {douments_str}"""
    if verbose:
      print(f"llm_summary_promt {llm_summary_promt}")
      print("**********************\n")
    response=self.chat_llm.invoke(llm_summary_promt)
    if verbose:
      print(f"response {response.content}")
    return response.content

**VectorStore** is a class for managing vector storage and retrieval using a language model-based embedding function.
    This class interfaces with a database to add documents and perform similarity searches based on queries.
    

In [6]:
class VectorStore:
  def __init__(self,emabdding_llm,db):
    self.emabdding_llm=emabdding_llm
    self.db=db
    """
    self.vector_store = Chroma(
        collection_name="rag_collection",
        embedding_function=emabdding_llm,
        persist_directory="./chroma_rag_db",)
    """

  def add_documents(self,documents:List[Document]):
    uuids = [str(uuid4()) for _ in range(len(documents))]
    self.db.add_documents(documents=documents, ids=uuids)

  def query(self,query:str,no_of_docs:int):
    return self.db.similarity_search(query,k=no_of_docs)

RAG is a wrapper class that implements the Retrieval-Augmented Generation (RAG) approach.
    This class is responsible for extracting text and images from PDF files,
    summarizing the content using a summarizer, storing the documents in a vector store,
    and querying the stored documents to retrieve relevant information based on user queries.
   

In [7]:

class RAG:
  def __init__(self,
               vector_store:VectorStore,
               summarizer:Summarizer,
               extractor:Extractor):
    self.extractor=extractor
    self.summarizer=summarizer
    self.vector_store=vector_store

  def extract(self,pdf_file:str,verbose=True):
    list_of_extensions =['.txt','.png']
    file_map=self.extractor.extract_pdf_content_and_list_files(pdf_file,list_of_extensions)
    extracted_text_file=file_map.get('.txt',[])[0]
    extracted_image_files=file_map.get('.png',[])
    if verbose:
      print(f"extracted_text_file {extracted_text_file}")
      print(f"extracted_image_files {extracted_image_files}")
    return extracted_text_file,extracted_image_files


  def summarize_text_files(self,textfile,shouldSummarizeText=True,hasLimittedLLMAccess=True,verbose=False):
    documents = self.extractor.extract_text_file_into_chuncks(textfile)
    if shouldSummarizeText:
      documents = self.summarizer.generate_text_summaries(documents,hasLimittedLLMAccess,verbose)
    if verbose:
      print(documents)
    return documents

  def summarize_image_files(self,imagefiles,hasLimittedLLMAccess=True,verbose=False):
    documents = self.summarizer.generate_image_summaries(imagefiles,hasLimittedLLMAccess,verbose)
    if verbose:
      print(documents)
    return documents

  def summarize(self,textfile,imagefiles,hasLimittedLLMAccess=True,shouldSummarizeText=True,verbose=False):
    text_documents = self.summarize_text_files(textfile,shouldSummarizeText,hasLimittedLLMAccess,verbose)
    if hasLimittedLLMAccess and shouldSummarizeText:
      print("delaying 70 sec due to LLM resource limitations")
      time.sleep(70)
    image_documents = self.summarize_image_files(imagefiles,hasLimittedLLMAccess,verbose)
    return text_documents,image_documents

  def save(self,text_document,image_document):
    self.vector_store.add_documents(text_document)
    self.vector_store.add_documents(image_document)

  def query_db(self,query:str,no_of_docs:int,verbose=False):
    documents = self.vector_store.query(query,no_of_docs)
    if verbose:
      print("Simillarity docs -")
      for document in documents:
        print(document.page_content)
        print("**********************\n")
    return documents

  def execute(self,query:str,no_of_docs:int=2,verbose=False)->str:
    simillarity_docs = self.query_db(query,no_of_docs,verbose)
    return self.summarizer.summarize(query,simillarity_docs,verbose)

Initializing embadding , chat , vision model & in memeory croma db vector store

In [8]:
LLM_API_KEY=userdata.get('GOOGLE_API_KEY') # should use your own key here
EMBEDING_MODEL_NAME='models/embedding-001'
CHAT_MODEL_NAME='gemini-pro'
VISION_MODEL_NAME='gemini-1.5-flash'


empty_response=RunnableLambda(
      lambda x:AIMessage(content="Error Processing document")
  )

emabdding_model = GoogleGenerativeAIEmbeddings(google_api_key=LLM_API_KEY,model=EMBEDING_MODEL_NAME)

chat_model = ChatGoogleGenerativeAI(model=CHAT_MODEL_NAME,
                                 google_api_key=LLM_API_KEY,
                                 temperature=0,
                                 max_output_tokens=1024).with_fallbacks([empty_response])
vision_model = ChatGoogleGenerativeAI(model=VISION_MODEL_NAME,
                                 google_api_key=LLM_API_KEY,
                                 temperature=0,
                                 max_output_tokens=1024).with_fallbacks([empty_response])
db = Chroma(
        collection_name="rag_collection",
        embedding_function=emabdding_model)


Initializion of RAG wraper class

In [9]:
rag=RAG(VectorStore(emabdding_model,db),Summarizer(chat_model,vision_model),Extractor(FileUtil()))

Absulute location of PDF file (which has mutiple Images embaded) on which symentic search has to be exceuted.

In [10]:
pdf_file = '/content/drive/MyDrive/datasets/pdfs/Clouded_Judgement.pdf'

Below piece of code will


1.   Extracts

  a. PDF text content into a text file

  b. each embaded imgage into a png file

2. Creates multiple chunks from the text file   
3. Optional - Summarize each chunks using chat model

4. Summarize each image png file vision model
5. Finally summaries of (each text chunks + each image file) are stored in Chroma DB for indexing   


In [ ]:
file_map=rag.extract(pdf_file)
document_map=rag.summarize(file_map[0],file_map[1])
rag.save(document_map[0],document_map[1])

Exectues symentic dense search in Chroma DB and summararization using chat model

In [93]:
rag.execute("what is the actual reported revenue of jamf ?",3)

'$142.6M'